<a href="https://colab.research.google.com/github/booogieeee/DarkMap/blob/main/ML_MAPPING_TOOL.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# Install dependencies!pip install earthaccess rasterio h5py numpy matplotlib -q

import earthaccess
import rasterio
import numpy as np
import matplotlib.pyplot as plt

print("All packages loaded ✓")

In [ ]:
auth = earthaccess.login(strategy="interactive")


In [ ]:
kampong_speu_bbox = (103.5, 10.9, 104.6, 11.8)

results = earthaccess.search_data(
    short_name="VNP46A2",
    temporal=("2023-01-01", "2023-12-31"),
    bounding_box=kampong_speu_bbox
)

print(f"Found {len(results)} granules")

In [ ]:
files = earthaccess.download(results[:5], local_path="./viirs_data/")
print("Downloaded:", files)

In [ ]:
import h5py
import numpy as np
import rasterio
from rasterio.transform import from_bounds
import os


hdf_files = [f for f in os.listdir("./viirs_data/") if f.endswith(".h5")]
print("Found files:", hdf_files)

In [ ]:
with h5py.File(f"./viirs_data/{hdf_files[0]}", "r") as f:

    grids = f["HDFEOS/GRIDS/"]
    print("Grids available:", list(grids.keys()))


    first_grid = list(grids.keys())[0]
    print("Data fields:", list(f[f"HDFEOS/GRIDS/{first_grid}/Data Fields/"].keys()))

In [ ]:
with h5py.File(f"./viirs_data/{hdf_files[0]}", "r") as f:

    ntl = f["HDFEOS/GRIDS/VIIRS_Grid_DNB_2d/Data Fields/Gap_Filled_DNB_BRDF-Corrected_NTL"][:]


    fill_value = f["HDFEOS/GRIDS/VIIRS_Grid_DNB_2d/Data Fields/Gap_Filled_DNB_BRDF-Corrected_NTL"].attrs.get("_FillValue", 65535)

    print("Array shape:", ntl.shape)
    print("Data type:", ntl.dtype)
    print("Min value:", ntl.min())
    print("Max value:", ntl.max())
    print("Fill value:", fill_value)

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

ntl_clean = np.where(ntl == -999.9, np.nan, ntl)

plt.figure(figsize=(10,10))
plt.imshow(ntl_clean, cmap="inferno", vmin=0, vmax=10)
plt.colorbar(label="Nighttime Light Radiance")
plt.title("VIIRS Nighttime Lights - Cambodia Title")
plt.show()

In [ ]:

lat_min, lat_max = 10.9, 11.8
lon_min, lon_max = 103.5, 104.6


tile_lat_max = 20.0
tile_lat_min = 10.0
tile_lon_min = 100.0
tile_lon_max = 110.0


row_min = int((tile_lat_max - lat_max) / (tile_lat_max - tile_lat_min) * 2400)
row_max = int((tile_lat_max - lat_min) / (tile_lat_max - tile_lat_min) * 2400)
col_min = int((lon_min - tile_lon_min) / (tile_lon_max - tile_lon_min) * 2400)
col_max = int((lon_max - tile_lon_min) / (tile_lon_max - tile_lon_min) * 2400)

# Crop
kampong_speu = ntl_clean[row_min:row_max, col_min:col_max]

print(f"Cropped shape: {kampong_speu.shape}")
plt.figure(figsize=(8, 8))
plt.imshow(kampong_speu, cmap="inferno", vmin=0, vmax=10)
plt.colorbar(label="Nighttime Light Radiance")
plt.title("Kampong Speu - Nighttime Lights")
plt.show()

In [ ]:
from rasterio.transform import from_bounds
import rasterio
import numpy as np

# Define the geographic bounds of our cropped Kampong Speu area
lon_min, lat_min = 103.5, 10.9
lon_max, lat_max = 104.6, 11.8

# Create a transformation matrix that maps pixels to GPS coordinates
transform = from_bounds(lon_min, lat_min, lon_max, lat_max,
                        kampong_speu.shape[1], kampong_speu.shape[0])

# Save as GeoTIFF
output_path = "./viirs_data/kampong_speu_ntl.tif"

with rasterio.open(
    output_path,
    'w',
    driver='GTiff',        # file format
    height=kampong_speu.shape[0],
    width=kampong_speu.shape[1],
    count=1,               # number of bands (1 = grayscale)
    dtype='float32',
    crs='EPSG:4326',       # standard GPS coordinate system
    transform=transform
) as dst:
    dst.write(np.nan_to_num(kampong_speu, nan=-999.9), 1)

print(f"Saved to {output_path}")